# 02 — Retrieval & QA

Assumes the vector store is already populated — run `01_indexing.ipynb` first.

Uses **hybrid search + cross-encoder reranking** (`bge-reranker-base`).

Sections:
1. Imports & config
2. Initialise pipeline
3. Single interactive query
4. Query with metadata filters
5. Batch test — all categories
6. Summary statistics

## 1 — Imports & Config

In [ ]:
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

# Make the package importable when running notebooks from notebooks/
sys.path.insert(0, str(Path("__file__").resolve().parent.parent / "src"))

from scientific_rag.vectorstore.embedders import LocalEmbedder
from scientific_rag.vectorstore.indexer import VectorStore
from scientific_rag.retrieval.retriever_rerank import RAGPipelineWithReranking
from scientific_rag.config import ANSWER_PROMPT_PATH, QA_MODEL
from tests.test_questions_comprehensive import test_questions, categories

load_dotenv()

ANSWER_PROMPT = ANSWER_PROMPT_PATH.read_text(encoding="utf-8")

print(f"Total questions loaded: {len(test_questions)}")
print("\nQuestions per category:")
for cat, qs in categories.items():
    print(f"  {cat}: {len(qs)}")

## 2 — Initialise Pipeline

In [ ]:
embedder = LocalEmbedder()
store    = VectorStore(embedder=embedder, use_hybrid=True)
rag      = RAGPipelineWithReranking(store=store, answer_prompt=ANSWER_PROMPT, model=QA_MODEL)

info = store.collection_info()
print(f"\nPipeline ready.  Collection has {info.points_count} vectors.")

## 3 — Single Interactive Query

Edit `QUESTION` and re-run this cell.

In [ ]:
QUESTION           = "What methods are used to measure runway friction?"
TOP_K              = 10
CANDIDATE_MULT     = 3   # fetch TOP_K × CANDIDATE_MULT candidates before reranking

t0 = time.time()
result = rag.ask(QUESTION, top_k=TOP_K, show_chunks=True, candidate_multiplier=CANDIDATE_MULT)
elapsed = round(time.time() - t0, 2)

print(f"Question: {QUESTION}")
print(f"Time: {elapsed}s")
print("=" * 72)
print(result["answer"])

print("\nSources:")
for s in result["sources"]:
    print(f"  • [{round(s.get('rerank_score', 0), 3)}]  {s['citation']}")

print(f"\nChunks retrieved (after reranking):")
for i, c in enumerate(result["chunks"], 1):
    print(f"  [{i}] rerank={round(c.get('rerank_score', 0), 3)}  orig={round(c.get('original_score', c.get('score', 0)), 3)}  "
          f"stem={c['metadata'].get('stem', '')[:45]}")

## 4 — Query with Metadata Filters

Restrict retrieval to a specific document or year.  
Available filter keys: `stem`, `year`, `doc_type`, `authors`

In [ ]:
FILTERED_QUESTION = "What is the role of aggregate gradation in SMA performance?"
FILTERS           = {"stem": "014. AGGREGATE SPECIFICATIONS FOR STONE MASTIC ASPHALT (SMA)"}

t0 = time.time()
r = rag.ask(FILTERED_QUESTION, top_k=10, filters=FILTERS, show_chunks=True)
elapsed = round(time.time() - t0, 2)

print(f"Q      : {FILTERED_QUESTION}")
print(f"Filter : {FILTERS}")
print(f"Time   : {elapsed}s")
print("=" * 72)
print(r["answer"])

print("\nSources:")
for s in r["sources"]:
    print(f"  • {s['citation']}")

## 5 — Batch Test

Runs every question through the reranking pipeline.  
Each question prints its answer, retrieved sources, and key-term coverage.

In [ ]:
def run_question(q: dict, top_k: int = 5, candidate_mult: int = 2) -> dict:
    sep = "=" * 72
    print(f"\n{sep}")
    print(f"[{q['id']}]  {q['category']}  |  difficulty: {q.get('difficulty', '—')}")
    print(f"Q: {q['question']}")
    print(sep)

    t0 = time.time()
    r  = rag.ask(q["question"], top_k=top_k, show_chunks=True, candidate_multiplier=candidate_mult)
    elapsed = round(time.time() - t0, 2)

    docs   = sorted({c["metadata"].get("stem", "")[:40] for c in r["chunks"]})
    scores = [round(c.get("rerank_score", c.get("score", 0)), 3) for c in r["chunks"][:3]]

    print(f"\n  time   : {elapsed}s")
    print(f"  docs   : {docs}")
    print(f"  scores : {scores}")
    print(f"  answer : {r['answer']}")

    terms   = q.get("answer_should_include", [])
    hits    = [t for t in terms if t.lower() in r["answer"].lower()]
    if terms:
        print(f"\n  key terms ({len(terms)}): {terms}")
        print(f"  found ({len(hits)}): {hits}")

    return dict(
        id=q["id"],
        category=q["category"],
        elapsed=elapsed,
        docs=docs,
        answer=r["answer"],
        sources=r["sources"],
        terms_hit=len(hits),
        terms_total=len(terms),
    )

print("Helper defined.")

### 5.1 — Single Document (specific)

In [ ]:
cat1 = [q for q in test_questions if q["category"] == "single_document_detailed"]
results_cat1 = [run_question(q, top_k=10) for q in cat1]

### 5.2 — Multi-document

In [ ]:
cat2 = [q for q in test_questions if q["category"] == "multi_document_integration"]
results_cat2 = [run_question(q, top_k=7) for q in cat2]

### 5.3 — Comparative Analysis

In [ ]:
cat3 = [q for q in test_questions if q["category"] == "comparative_analysis"]
results_cat3 = [run_question(q, top_k=10) for q in cat3]

### 5.4 — Technical Terminology

In [ ]:
cat4 = [q for q in test_questions if q["category"] == "technical_terminology"]
results_cat4 = [run_question(q, top_k=10) for q in cat4]

### 5.5 — Operational / Design / Methodology

In [ ]:
cat567 = [q for q in test_questions if q["category"] in
          ("operational_complexity", "design_construction", "methodology")]
results_cat567 = [run_question(q, top_k=10) for q in cat567]

### 5.6 — Out of Scope (negative test)

These questions should **not** be answered from the corpus.  
A low rerank score confirms the system correctly finds no relevant content.

In [ ]:
cat8 = [q for q in test_questions if q["category"] == "out_of_scope"]

results_cat8 = []
for q in cat8:
    print(f"\n{'='*72}")
    print(f"[{q['id']}] {q['question']}")

    r = rag.ask(q["question"], top_k=3, show_chunks=True)
    top_score = max((c.get("rerank_score", c.get("score", 0)) for c in r["chunks"]), default=0)

    print(f"  top rerank score : {round(top_score, 3)}  (low = correctly filtered)")
    print(f"  answer           : {r['answer'][:300]}")
    results_cat8.append({"id": q["id"], "top_score": round(top_score, 3)})

### 5.7 — Ambiguous / Complex Reasoning

In [ ]:
cat910 = [q for q in test_questions if q["category"] in ("ambiguous", "complex_reasoning")]
results_cat910 = [run_question(q, top_k=7) for q in cat910]

## 6 — Summary

In [ ]:
all_results = (
    results_cat1 + results_cat2 + results_cat3 +
    results_cat4 + results_cat567 + results_cat910
)

avg_t        = round(sum(r["elapsed"]     for r in all_results) / len(all_results), 2)
total_terms  = sum(r["terms_total"] for r in all_results)
total_hits   = sum(r["terms_hit"]   for r in all_results)
coverage_pct = round(total_hits / total_terms * 100) if total_terms else 0

print("=" * 72)
print("OVERALL SUMMARY")
print("=" * 72)
print(f"  Questions run     : {len(all_results)}")
print(f"  Avg response time : {avg_t}s")
print(f"  Key-term coverage : {total_hits}/{total_terms}  ({coverage_pct}%)")
print("=" * 72)

print(f"\n{'ID':<22} {'category':<28} {'time':>6} {'terms':>7}")
print("-" * 72)
for r in all_results:
    denom = r['terms_total'] if r['terms_total'] else 1
    print(f"  {r['id']:<20} {r['category']:<28} {r['elapsed']:>5}s  {r['terms_hit']:>2}/{denom}")